# Experiment 7: Data Loading, Storage and File Formats
## Problem Statement: Analyzing Sales Data from Multiple File Formats

**Aim:** Load sales data from CSV, Excel and JSON files, inspect data quality, clean and transform the data, analyze sales performance, and visualize the results.

**Dataset used:** `Sales.csv`, `Sales.xlsx`, and `Sales.json`. These are three copies of the same sales dataset in different file formats. Therefore, they will be compared and loaded separately, but their rows will not be appended together (that would duplicate the same sales records).

## 1. Import required libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

## 2. Load data from CSV, Excel and JSON

In [ ]:
csv_df = pd.read_csv("Sales.csv")
excel_df = pd.read_excel("Sales.xlsx")
json_df = pd.read_json("Sales.json")

print("CSV shape:", csv_df.shape)
print("Excel shape:", excel_df.shape)
print("JSON shape:", json_df.shape)

display(csv_df.head())
display(excel_df.head())
display(json_df.head())

## 3. Explore the structure and identify data quality issues

In [ ]:
print("Column names:")
print(csv_df.columns.tolist())

print("\\nData types:")
print(csv_df.dtypes)

print("\\nMissing values:")
display(csv_df.isnull().sum())

print("Exact duplicate rows:", csv_df.duplicated().sum())

print("\\nNumeric summary:")
display(csv_df.describe(include="all").T)

## 4. Convert the loaded data into a unified dataframe

The three files contain the same records. We use the CSV dataframe as the main dataframe after checking that all three formats have the same dimensions and columns. This avoids counting every sale three times.

In [ ]:
# Check that all three files have matching columns and row counts
same_structure = (
    list(csv_df.columns) == list(excel_df.columns) == list(json_df.columns)
    and csv_df.shape == excel_df.shape == json_df.shape
)
print("All three files have the same structure:", same_structure)

# Create one working dataframe from the CSV file
df = csv_df.copy()

# Standardize column names by removing extra spaces
df.columns = df.columns.str.strip()

print("Unified dataframe shape:", df.shape)
display(df.head())

## 5. Clean the data

In [ ]:
# Remove extra spaces from text columns
text_columns = df.select_dtypes(include="object").columns
for column in text_columns:
    df[column] = df[column].astype("string").str.strip()

# Remove exact duplicate rows
duplicate_count = df.duplicated().sum()
df = df.drop_duplicates().copy()
print("Duplicate rows removed:", duplicate_count)

# Standardize category names and correct a common spelling variation
df["Category"] = df["Category"].str.strip().str.title()
df["Category"] = df["Category"].replace({"Electronic": "Electronics"})

# Convert dates. The source contains mixed date styles; invalid text becomes NaT.
df["Order_Date"] = pd.to_datetime(
    df["Order_Date"], format="mixed", errors="coerce", dayfirst=False
)

# Convert numeric columns. Remove currency symbols/commas before conversion.
for column in ["Quantity", "Price", "Total"]:
    cleaned_values = (
        df[column].astype("string")
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
    )
    df[column] = pd.to_numeric(cleaned_values, errors="coerce")

print("Missing values after initial conversion:")
display(df.isnull().sum())

### Handle missing values

- Missing category values are labelled `Unknown`.
- Missing quantity and price values are filled with their respective medians.
- Missing totals are calculated using `Quantity × Price`.
- Invalid dates are retained as missing (`NaT`) rather than inventing a date.
- Negative quantity or price values are flagged for review, not silently deleted, because the source does not explain whether they represent returns or data-entry errors.

In [ ]:
df["Category"] = df["Category"].fillna("Unknown")

quantity_median = df["Quantity"].median()
price_median = df["Price"].median()

df["Quantity"] = df["Quantity"].fillna(quantity_median)
df["Price"] = df["Price"].fillna(price_median)

missing_total = df["Total"].isna()
df.loc[missing_total, "Total"] = (
    df.loc[missing_total, "Quantity"] * df.loc[missing_total, "Price"]
)

# Flag unusual negative values for inspection
df["Review_Negative_Value"] = (df["Quantity"] < 0) | (df["Price"] < 0) | (df["Total"] < 0)

print("Missing values after cleaning:")
display(df.isnull().sum())
print("Rows with negative values flagged for review:", df["Review_Negative_Value"].sum())
display(df.loc[df["Review_Negative_Value"], ["ID", "Product", "Quantity", "Price", "Total", "Status"]])

## 6. Transform the data and derive new variables

In [ ]:
# Derive year and month from valid order dates
df["Order_Year"] = df["Order_Date"].dt.year
df["Order_Month"] = df["Order_Date"].dt.month_name()
df["Order_Month_Number"] = df["Order_Date"].dt.month

# Calculate sales value per unit/order from the cleaned total
df["Average_Item_Value"] = df["Total"] / df["Quantity"].replace(0, np.nan)

display(df.head())

**Note on merging:** The CSV, Excel and JSON files are alternate-format copies of the same dataset, not separate datasets. Merging or concatenating them would duplicate every sale. The transformation task is therefore demonstrated by deriving year, month and average item value columns.

## 7. Descriptive statistics and sales analysis

In [ ]:
print("Number of records:", len(df))
print("Number of unique orders:", df["Order_ID"].nunique())
print("Total sales:", round(df["Total"].sum(), 2))
print("Average order value:", round(df["Total"].sum() / df["Order_ID"].nunique(), 2))

print("\\nDescriptive statistics:")
display(df[["Quantity", "Price", "Total"]].describe())

print("\\nSales by product category:")
category_sales = df.groupby("Category")["Total"].sum().sort_values(ascending=False)
display(category_sales.to_frame("Total_Sales"))

print("\\nOrders by status:")
status_counts = df["Status"].value_counts()
display(status_counts.to_frame("Number_of_Orders"))

print("\\nSales by payment method:")
payment_sales = df.groupby("Payment_Method")["Total"].sum().sort_values(ascending=False)
display(payment_sales.to_frame("Total_Sales"))

## 8. Visualize the sales data

In [ ]:
# Bar plot: sales by category
plt.figure(figsize=(9, 5))
category_sales.plot(kind="bar")
plt.title("Total Sales by Product Category")
plt.xlabel("Product Category")
plt.ylabel("Total Sales")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Pie chart: order status distribution
plt.figure(figsize=(7, 7))
status_counts.plot(kind="pie", autopct="%1.1f%%", startangle=90)
plt.title("Order Status Distribution")
plt.ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
# Box plot: distribution of order totals
plt.figure(figsize=(8, 5))
df.boxplot(column="Total")
plt.title("Distribution of Order Totals")
plt.ylabel("Total")
plt.tight_layout()
plt.show()

## 9. Save the cleaned data in multiple formats

In [ ]:
df.to_csv("Cleaned_Sales.csv", index=False)
df.to_excel("Cleaned_Sales.xlsx", index=False)
df.to_json("Cleaned_Sales.json", orient="records", date_format="iso", indent=4)

print("Cleaned data saved in CSV, Excel and JSON formats.")

## 10. Conclusion

In [ ]:
print("The sales dataset was loaded from CSV, Excel and JSON formats.")
print("The data was explored, cleaned, standardized and transformed.")
print("Sales were analyzed using descriptive statistics and group-wise aggregation.")
print("Bar, pie and box plots were created to visualize the results.")
print("The cleaned dataframe was exported into CSV, Excel and JSON formats.")